In [1]:
from pyspark.sql import functions as F

silver_df = spark.table("silver.meter_readings")

silver_count = silver_df.count()

print("SILVER QA")
print("_" * 50)
print(f"Silver rows: {silver_count}") 

StatementMeta(, 51d27179-4de6-466f-afec-9021aa9c35c8, 3, Finished, Available, Finished, False)

SILVER QA
__________________________________________________
Silver rows: 999283


In [2]:
print("SILVER SCHEMA")
print("_" * 50)

silver_df.printSchema()

StatementMeta(, 51d27179-4de6-466f-afec-9021aa9c35c8, 4, Finished, Available, Finished, False)

SILVER SCHEMA
__________________________________________________
root
 |-- HouseholdID: string (nullable = true)
 |-- TariffType: string (nullable = true)
 |-- ReadingTimestamp: timestamp (nullable = true)
 |-- ConsumptionKWh: double (nullable = true)
 |-- ReadingDate: date (nullable = true)
 |-- ReadingYear: integer (nullable = true)
 |-- ReadingMonth: integer (nullable = true)
 |-- ReadingDay: integer (nullable = true)
 |-- ReadingHour: integer (nullable = true)
 |-- DayOfWeek: string (nullable = true)



In [3]:
print("NULL COUNTS")
print("_" * 50 )

null_counts = silver_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in silver_df.columns
])

display(null_counts)

StatementMeta(, 51d27179-4de6-466f-afec-9021aa9c35c8, 5, Finished, Available, Finished, False)

NULL COUNTS
__________________________________________________


SynapseWidget(Synapse.DataFrame, 451f2d92-7693-4d92-b0ba-683070ba0974)

In [4]:
duplicate_keys = (
    silver_df
    .groupBy("HouseholdID", "ReadingTimestamp")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_key_count = duplicate_keys.count()

print("DUPLICATE KEY CHECK")
print("_" * 50)
print(f"Duplicate Household/Timestamp Keys: {duplicate_key_count:}")

StatementMeta(, 51d27179-4de6-466f-afec-9021aa9c35c8, 6, Finished, Available, Finished, False)

DUPLICATE KEY CHECK
__________________________________________________
Duplicate Household/Timestamp Keys: 0


In [5]:
invalid_consumptio_count = (
    silver_df.filter(
        F.col("ConsumptionKWh").isNull() |
        (F.col("ConsumptionKWh") <0)
    )
    .count()
)

print("CONSUMPTION CHECK")
print("_" * 50)
print(f"Invalid consumption rows: {invalid_consumptio_count:,}")

StatementMeta(, 51d27179-4de6-466f-afec-9021aa9c35c8, 7, Finished, Available, Finished, False)

CONSUMPTION CHECK
__________________________________________________
Invalid consumption rows: 0


In [6]:
print("TARIFF VALUES")
print("-" * 50)

silver_df.groupBy("TariffType").count().orderBy(
    F.desc("count")
).show(truncate=False)

StatementMeta(, 51d27179-4de6-466f-afec-9021aa9c35c8, 8, Finished, Available, Finished, False)

TARIFF VALUES
--------------------------------------------------
+----------+------+
|TariffType|count |
+----------+------+
|Std       |999283|
+----------+------+



In [7]:
print("DATE RANGE")
print("-" * 50)

silver_df.select(
    F.min("ReadingTimestamp").alias("MinReadingTimestamp"),
    F.max("ReadingTimestamp").alias("MaxReadingTimestamp")
).show(truncate=False)

StatementMeta(, 51d27179-4de6-466f-afec-9021aa9c35c8, 9, Finished, Available, Finished, False)

DATE RANGE
--------------------------------------------------
+-------------------+-------------------+
|MinReadingTimestamp|MaxReadingTimestamp|
+-------------------+-------------------+
|2011-12-06 13:00:00|2014-02-28 00:00:00|
+-------------------+-------------------+



In [8]:
rejected_df = spark.table("silver.rejected_meter_readings")

print("REJECTION TABLE")
print("-" * 50)
print(f"Rejected rows: {rejected_df.count():,}")

rejected_df.groupBy("RejectionReason").count().show(
    truncate=False
)

StatementMeta(, 51d27179-4de6-466f-afec-9021aa9c35c8, 10, Finished, Available, Finished, False)

REJECTION TABLE
--------------------------------------------------
Rejected rows: 29
+-------------------+-----+
|RejectionReason    |count|
+-------------------+-----+
|INVALID_CONSUMPTION|29   |
+-------------------+-----+

